# Generate Morphology Feature Descriptions

This notebook creates a feature description table by:
1. Loading the original feature descriptions from Supp_Data_Table_5
2. Renaming features using the morpho_feature_rename.csv mapping
3. Checking for missing features in both directions
4. Outputting a clean CSV for use in the main feature table

In [1]:
import pandas as pd
import numpy as np

## Load source data

In [2]:
# Load the original feature descriptions
supp_data = pd.read_csv('/root/capsule/results/Supp_Data_Table_5_Morphology_Feature_Names_And_Description_RFC_Params(Supp_Data_Table_5_Morphology_Fe).csv', encoding='latin-1')
print(f"Original feature descriptions: {len(supp_data)} rows")
print(f"Columns: {list(supp_data.columns)}")
supp_data.head()

Original feature descriptions: 94 rows
Columns: ['feature_name', 'skeleton_keys_name', 'description', 'compartment', 'data_source']


,feature_name,skeleton_keys_name,description,compartment,data_source
0,apical horiz. bias,apical_dendrite_bias_x,Unsigned difference in extent in x in one dir...,dendrites,upright swc
1,apical vert. bias,apical_dendrite_bias_y,Signed difference in extent in y in one direct...,dendrites,upright swc
2,apical early branch path,apical_dendrite_early_branch_path,Ratio of the maximum length of all the ï¿½shor...,dendrites,upright swc
3,apical width,apical_dendrite_extent_x,Total extent in the x-direction (micron),dendrites,upright swc
4,apical height,apical_dendrite_extent_y,Total extent in the y-direction (micron),dendrites,upright swc


In [3]:
# Load the feature rename mapping
rename_dict = pd.read_csv('/root/capsule/data/formatting_dicts/morpho_feature_rename.csv')
print(f"Rename dictionary: {len(rename_dict)} rows")
print(f"Columns: {list(rename_dict.columns)}")
rename_dict.head()

Rename dictionary: 69 rows
Columns: ['old_feature_name', 'new_feature_name', 'type']


,old_feature_name,new_feature_name,type
0,apical_dendrite_bias_x,Horizontal apical bias,apical
1,apical_dendrite_bias_y,Vertical apical bias,apical
2,apical_dendrite_depth_pc_0,Apical depth pc 0,apical
3,apical_dendrite_depth_pc_1,Apical depth pc 1,apical
4,apical_dendrite_depth_pc_2,Apical depth pc 2,apical


## Rename features

In [4]:
# Create a mapping from old_feature_name (skeleton_keys_name) to new_feature_name
rename_mapping = dict(zip(rename_dict['old_feature_name'], rename_dict['new_feature_name']))
print(f"Mapping contains {len(rename_mapping)} entries")

Mapping contains 69 entries


In [5]:
# Apply the renaming
feature_descriptions = supp_data.copy()
feature_descriptions['feature_name'] = feature_descriptions['skeleton_keys_name'].map(rename_mapping)

# Check for unmapped features (features in supp_data not in rename dict)
unmapped = feature_descriptions[feature_descriptions['feature_name'].isna()]
print(f"\n{'='*60}")
print(f"Features in Supp_Data_Table_5 NOT in rename dict: {len(unmapped)}")
print(f"{'='*60}")
if len(unmapped) > 0:
    print("\nThese features will need manual descriptions or should be removed:")
    for idx, row in unmapped.iterrows():
        print(f"  - {row['skeleton_keys_name']}: {row.get('description', 'N/A')[:60]}...")


Features in Supp_Data_Table_5 NOT in rename dict: 25

These features will need manual descriptions or should be removed:
  - dendrite_frac_above_axon: % of basal dendrite nodes above the full y-extent of the axo...
  - dendrite_frac_below_axon: % of basal dendrite nodes below the full y-extent of the axo...
  - dendrite_frac_intersect_axon: % of basal dendrite nodes that overlap with the full y-exten...
  - axon_frac_above_dendrite: % of axon nodes above the full y-extent of the dendrite node...
  - axon_frac_below_dendrite: % of axon nodes below the full y-extent of the dendritenodes...
  - axon_frac_intersect_dendrite: % of axon nodes that overlap with the full y-extent of the d...
  - axon_emd_with_dendrite: Earth moverï¿½s distance metric calculated between the norma...
  - axon_depth: Medial-lateral span of the complete axon arbor...
  - axon_height: Dorsal-lateral span of the complete axon arbor...
  - axon_width: Anterior-posterior span of the complete axon arbor...
  - complet

In [6]:
# Check for features in rename dict that don't have descriptions
skeleton_keys_in_supp = set(supp_data['skeleton_keys_name'])
old_features_in_rename = set(rename_dict['old_feature_name'])
missing_descriptions = old_features_in_rename - skeleton_keys_in_supp

print(f"\n{'='*60}")
print(f"Features in rename dict NOT in Supp_Data_Table_5: {len(missing_descriptions)}")
print(f"{'='*60}")
if len(missing_descriptions) > 0:
    print("\nThese features need descriptions to be added manually:")
    for feature in sorted(missing_descriptions):
        new_name = rename_mapping[feature]
        print(f"  - {feature} -> '{new_name}'")


Features in rename dict NOT in Supp_Data_Table_5: 0


## Save output

In [7]:
# Remove rows with unmapped features (if any)
feature_descriptions_clean = feature_descriptions.dropna(subset=['feature_name'])
print(f"\nFinal feature descriptions: {len(feature_descriptions_clean)} rows")
print(f"Removed {len(feature_descriptions) - len(feature_descriptions_clean)} rows with unmapped features")


Final feature descriptions: 69 rows
Removed 25 rows with unmapped features


In [8]:
# Reorder columns to match the desired format
output_df = feature_descriptions_clean[['feature_name', 'skeleton_keys_name', 'description', 'compartment', 'data_source']]
output_df.head()

,feature_name,skeleton_keys_name,description,compartment,data_source
0,Horizontal apical bias,apical_dendrite_bias_x,Unsigned difference in extent in x in one dir...,dendrites,upright swc
1,Vertical apical bias,apical_dendrite_bias_y,Signed difference in extent in y in one direct...,dendrites,upright swc
2,Early apical branch path,apical_dendrite_early_branch_path,Ratio of the maximum length of all the ï¿½shor...,dendrites,upright swc
3,Horizontal apical extent,apical_dendrite_extent_x,Total extent in the x-direction (micron),dendrites,upright swc
4,Vertical apical extent,apical_dendrite_extent_y,Total extent in the y-direction (micron),dendrites,upright swc


In [9]:
# Save to CSV
output_path = '/root/capsule/scratch/20260807/tables/morphology_feature_description.csv'
output_df.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")


Saved to: /root/capsule/scratch/20260807/tables/morphology_feature_description.csv


In [10]:
output_df

,feature_name,skeleton_keys_name,description,compartment,data_source
0,Horizontal apical bias,apical_dendrite_bias_x,Unsigned difference in extent in x in one dir...,dendrites,upright swc
1,Vertical apical bias,apical_dendrite_bias_y,Signed difference in extent in y in one direct...,dendrites,upright swc
2,Early apical branch path,apical_dendrite_early_branch_path,Ratio of the maximum length of all the ï¿½shor...,dendrites,upright swc
3,Horizontal apical extent,apical_dendrite_extent_x,Total extent in the x-direction (micron),dendrites,upright swc
4,Vertical apical extent,apical_dendrite_extent_y,Total extent in the y-direction (micron),dendrites,upright swc
...,...,...,...,...,...
70,Axon depth pc 0,axon_depth_pc_0,First principal component that exceeds 5% expl...,axon,layer aligned swc
71,Axon depth pc 1,axon_depth_pc_1,Second principal component that exceeds 5% exp...,axon,layer aligned swc
72,Axon depth pc 2,axon_depth_pc_2,Third principal component that exceeds 5% expl...,axon,layer aligned swc
73,Axon depth pc 3,axon_depth_pc_3,Fourth principal component that exceeds 5% exp...,axon,layer aligned swc
